# 🔮 Chandra OCR 2 — PDF OCR on Google Colab

This notebook runs **Datalab Chandra OCR 2** to extract text from image-based PDFs.

**Setup:** Runtime → Change runtime type → **T4 GPU** (free tier works)

Chandra 2 is a 4B-parameter OCR model outputting structured **Markdown / HTML / JSON** while preserving layout, tables, math, and handwriting.

> ⚠️ License: Free for research, personal use, and startups under $2M funding/revenue.

## 1 — Check GPU

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → GPU")

Mon Apr 20 02:01:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 2 — Install dependencies

In [5]:
# Install chandra-ocr with HuggingFace backend (flash-attn is optional, skip it)
!pip install -q "chandra-ocr[hf]"

# pdf2image for converting PDF pages to images
!pip install -q pdf2image
!apt-get -qq install -y poppler-utils

## 3 — Upload your PDF

In [3]:
from google.colab import files
import os

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
pdf_path = os.path.join("/content", pdf_filename)
print(f"\n✅ Uploaded: {pdf_filename} ({os.path.getsize(pdf_path) / 1e6:.1f} MB)")

Saving Level1-Books104_text.pdf to Level1-Books104_text (1).pdf

✅ Uploaded: Level1-Books104_text (1).pdf (20.8 MB)


## 4 — Convert PDF pages to images

In [4]:
import pypdfium2 as pdfium

doc = pdfium.PdfDocument(pdf_path)
total_pages = len(doc)
doc.close()
print(f"✅ PDF has {total_pages} pages")

✅ PDF has 253 pages


## 5 — Load Chandra OCR 2 model

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch

MODEL_NAME = "datalab-to/chandra-ocr-2"

print(f"Loading {MODEL_NAME}... (takes a few minutes on first run)")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
model.processor = AutoProcessor.from_pretrained(MODEL_NAME)
model.processor.tokenizer.padding_side = "left"

print("✅ Model loaded!")

Loading datalab-to/chandra-ocr-2... (takes a few minutes on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

✅ Model loaded!


## 6 — Run OCR on all pages

In [ ]:
from pdf2image import convert_from_path
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown
import gc, torch, warnings

warnings.filterwarnings("ignore", message=".*processor_kwargs.*")

all_markdown = []

for i in range(total_pages):
    print(f"Page {i+1}/{total_pages}...", end=" ")

    # Lower DPI + shrink to fit T4 VRAM
    page_img = convert_from_path(
        pdf_path, dpi=150, first_page=i+1, last_page=i+1
    )[0]

    # Cap the longest side at 1024px
    max_side = max(page_img.size)
    if max_side > 1024:
        scale = 1024 / max_side
        page_img = page_img.resize(
            (int(page_img.width * scale), int(page_img.height * scale))
        )

    batch = [BatchInputItem(image=page_img, prompt_type="ocr_layout")]

    with torch.no_grad():
        result = generate_hf(batch, model)[0]

    md = parse_markdown(result.raw)
    all_markdown.append(md)

    del page_img, batch, result
    gc.collect()
    torch.cuda.empty_cache()

    print(f"✅ ({len(md)} chars)")

full_text = "\n\n---\n\n".join(
    [f"## Page {i+1}\n\n{md}" for i, md in enumerate(all_markdown)]
)
print(f"\n🎉 Done! {len(full_text)} chars across {total_pages} pages")

Page 1/253... 

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


✅ (390 chars)
Page 2/253... 

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


✅ (3606 chars)
Page 3/253... 

...

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


✅ (2703 chars)
Page 9/253... 

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


✅ (1412 chars)
Page 10/253... 

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


## 7 — View results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(full_text))

## 8 — Save & download

In [ ]:
output_name = os.path.splitext(pdf_filename)[0]
md_path = f"/content/{output_name}_ocr.md"
txt_path = f"/content/{output_name}_ocr.txt"

with open(md_path, "w", encoding="utf-8") as f:
    f.write(full_text)

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"Saved: {md_path}")
print(f"Saved: {txt_path}")

files.download(md_path)
files.download(txt_path)
print("\n✅ Downloaded!")

## 9 — (Optional) View a specific page

In [ ]:
PAGE_NUMBER = 1  # change this (1-indexed)

if 1 <= PAGE_NUMBER <= len(all_markdown):
    print(f"=== Page {PAGE_NUMBER} ===")
    display(Markdown(all_markdown[PAGE_NUMBER - 1]))
else:
    print(f"Invalid. Document has {len(all_markdown)} pages.")

## 10 — (Optional) Process specific page range

If your PDF is large and you hit memory limits, process a subset.

In [ ]:
START_PAGE = 1
END_PAGE = 5  # change as needed

subset_pages = pages[START_PAGE - 1 : END_PAGE]

for i, page_img in enumerate(subset_pages):
    page_num = START_PAGE + i
    print(f"Processing page {page_num}...", end=" ")
    batch = [BatchInputItem(image=page_img, prompt_type="ocr_layout")]
    result = generate_hf(batch, model)[0]
    md = parse_markdown(result.raw)
    print("✅")
    display(Markdown(f"### Page {page_num}\n\n{md}"))